For licensing see accompanying LICENSE file.  
Copyright (C) 2025 Apple Inc. All Rights Reserved.

# Exploring Semantic Regex Abstraction
Semantic regexes encode feature complexity. Here we explore how feature complexity varies across the model, using only the semantic regexes.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import re
import json
import brotli
import pandas as pd
import altair as alt
from pathlib import Path
alt.data_transformers.enable("vegafusion")



DataTransformerRegistry.enable('vegafusion')

## Data Loading

In [ ]:
def import_feature_json(json_file_path):
    """
    Import a feature JSON file (brotli compressed) containing multiple features grouped by layer.

    Parameters:
        json_file_path (str): Path to the brotli compressed JSON file

    Returns:
        pd.DataFrame: DataFrame with all features from the file
    """
    # Read and decompress brotli file
    with open(json_file_path, 'rb') as f:
        compressed_data = f.read()
        decompressed_data = brotli.decompress(compressed_data)
        group_content = json.loads(decompressed_data)

    rows = []

    # Iterate through each feature in the grouped file
    for feature_key, content in group_content.items():
        # Extract basic feature information
        row = {
            'model_id': content['feature']['model_id'],
            'layer': content['feature']['layer'],
            'index': content['feature']['index'],
            'description': content['description']['description'],
            'method': content['description'].get('method'),
        }

        # Extract evaluation metrics (just the values)
        evaluation = content['evaluation']
        for metric_name, metric_data in evaluation.items():
            row[f'{metric_name}'] = metric_data['value']

        rows.append(row)

    # Create DataFrame with all rows from this file
    df = pd.DataFrame(rows)
    return df

In [14]:
def import_all_feature_jsons_multi_folder(base_path, folder_pattern='*'):
    """
    Import all feature brotli files from multiple folders into a single pandas DataFrame.

    Parameters:
        base_path (str): Path to the base directory containing multiple folders
        folder_pattern (str): Pattern to match folder names (default '*' for all folders)

    Returns:
        pd.DataFrame: Combined DataFrame with all features
    """
    base_path = Path(base_path)
    all_dfs = []

    # Loop through all matching folders
    for folder in base_path.glob(folder_pattern):
        if folder.is_dir():
            print(f"\nProcessing folder: {folder.name}")

            # Loop through all brotli files in the current folder
            for brotli_file in folder.glob('*.json.brotli'):
                try:
                    df = import_feature_json(brotli_file)
                    # Add folder name and filename to all rows from this file
                    df['folder_name'] = folder.name
                    df['filename'] = brotli_file.name
                    all_dfs.append(df)
                    print(f"  Processed: {brotli_file.name} ({len(df)} features)")
                except Exception as e:
                    print(f"  Error processing {brotli_file.name}: {e}")

    if all_dfs:
        combined_df = pd.concat(all_dfs, ignore_index=True)
        print(f"\nSuccessfully imported {combined_df.shape[0]} features from {len(combined_df.folder_name.unique())} folders")
        return combined_df
    else:
        print("No brotli files found or processed successfully")
        return pd.DataFrame()

In [6]:
experiments_dir = os.path.join('../', 'artifacts', 'experiments')
experiment = 'cf1daa'
model_name_map = {
    'GPT2-RES-25k': 'gpt2-small_res-jb_generation-only',
    'Gemma-2-2B-RES-16k': 'gemma-2-2b_gemmascope-res-16k',
    'Gemma-2-2B-RES-65k': 'gemma-2-2b_gemmascope-res-65k',
}
model_dfs = {}
for model_name, model in model_name_map.items():
    model_dfs[model_name] = import_all_feature_jsons_multi_folder(experiments_dir, f'{experiment}_semantic_regex_{model}')


Processing folder: cf1daa_semantic_regex_gpt2-small_res-jb_generation-only
  Processed: layer-4.json.brotli (982 features)
  Processed: layer-8.json.brotli (998 features)
  Processed: layer-1.json.brotli (838 features)
  Processed: layer-2.json.brotli (806 features)
  Processed: layer-11.json.brotli (999 features)
  Processed: layer-7.json.brotli (1000 features)
  Processed: layer-0.json.brotli (802 features)
  Processed: layer-9.json.brotli (1000 features)
  Processed: layer-5.json.brotli (998 features)
  Processed: layer-6.json.brotli (1000 features)
  Processed: layer-10.json.brotli (1000 features)
  Processed: layer-3.json.brotli (854 features)

Successfully imported 11277 features from 1 folders

Processing folder: cf1daa_semantic_regex_gemma-2-2b_gemmascope-res-16k
  Processed: layer-4.json.brotli (1000 features)
  Processed: layer-12.json.brotli (999 features)
  Processed: layer-20.json.brotli (1000 features)
  Processed: layer-8.json.brotli (998 features)
  Processed: layer-17

## Semantic Regex Abstraction Plots

In [7]:
primitive_taxonomy = ['symbol', 'lexeme', 'field', 'context']
primitive_colors = [
    "#f2e5ff",
    "#c994c7",
    "#756bb1",
    "#4a1486"
]

combo_taxonomy = ['individual', 'optional', 'context', 'or', 'combination']
combo_colors = [
    "#ffd92f",   # orange

    # 4 shades of green (light → dark)
    "#e5f5e0",
    "#a1d99b",
    "#31a354",
    "#006d2c",
]

### Number of tags per layer

In [8]:
def count_tags(description, tags):
    """Count occurrences of special words in description"""
    if pd.isna(description):
        return {tag: 0 for tag in tag_counts}

    description_lower = str(description).lower()
    tag_counts = {}

    for tag in tags:
        # Use word boundaries for other words to avoid partial matches
        pattern = r'\b' + re.escape(tag.lower()) + r'\b'
        if tag == 'sequence':
            pattern = r"\]\s*\["
        elif tag == 'alternation':
            pattern = r"\|"
        elif tag == 'quantification':
            pattern = r"\?"
        elif tag == 'none':
            pattern = r'^\[:(?:(?!\[:|:\]).)*:\]$'
        tag_counts[tag] = len(re.findall(pattern, description_lower))

    return tag_counts

### Bar Plot

In [9]:
def plot_bar_chart(df, taxonomy, colors, normalize, show_x=True, show_y=True, legend_title='Component Type', legend=True):
    width = len(df['layer'].unique()) * 12

    # Apply the counting function to each row
    df = df.copy()
    tag_counts = df['description'].apply(lambda x: count_tags(x, taxonomy))

    # Add new columns for each special word count
    for tag in taxonomy:
        df[f'{tag}_count'] = tag_counts.apply(lambda x: x[tag])

    # Calculate total special words per row
    df['total_special_words'] = df[[f'{tag}_count' for tag in taxonomy]].sum(axis=1)
    df['layer_sort'] = df['layer'].str.extract('(\d+)').astype(int)
    df = df.sort_values('layer_sort')

    # Create a long-format dataframe for Altair visualization
    df_long = df.melt(
        id_vars=['layer_sort'],
        value_vars=[f'{word}_count' for word in taxonomy],
        var_name='word_type',
        value_name='count'
    )

    # Clean up the word_type column to remove '_count' suffix
    df_long['word_type'] = df_long['word_type'].str.replace('_count', '')

    if normalize:
        y_encoding_bar = alt.Y(
            'sum(count):Q',
            stack="normalize",
            title=f'% {legend_title}'  # default axis (labels/ticks/grid)
        )
    else:
        y_encoding_bar = alt.Y(
            'sum(count):Q',
            title='Count'  # default axis (labels/ticks/grid)
        )
    if not show_y:
        y_encoding_bar.axis = axis=alt.Axis(labels=False, ticks=False, title=None)

    x_encoding_bar = alt.X(
        'layer_sort:Q',
        title='Layer Index',
        sort='ascending',
        scale=alt.Scale(
            domain=[0, df_long['layer_sort'].max()],
            padding=5
        )
    )
    if not show_x:
        x_encoding_bar.axis = alt.Axis(labels=False, ticks=False, title=None)

    color = alt.Color('word_type:N',
                    title=legend_title,
                    sort=taxonomy[::-1],
                    scale=alt.Scale(range=colors[::-1])
                    )
    if not legend:
        color = color.legend(None)

    # Make a numeric sort key from taxonomy
    order_map = {w: i for i, w in enumerate(taxonomy)}
    df_long['taxonomy_order'] = df_long['word_type'].map(order_map)

    # Create the main stacked bar chart
    bar_chart = alt.Chart(df_long).mark_bar(width=10).add_selection(
        alt.selection_multi(fields=['word_type'])
    ).encode(
        x=x_encoding_bar,
        y=y_encoding_bar,
        color=color,
        order=alt.Order('taxonomy_order:Q', sort='ascending')
    ).properties(
        height=70,
        width=width,
    )

    return bar_chart

### Line Chart

In [10]:
def plot_line_chart(df, taxonomy, hide_y_axis=False):
    width = len(df['layer'].unique()) * 12

    # Apply the counting function to each row
    tag_counts = df['description'].apply(lambda x: count_tags(x, taxonomy))

    # Add new columns for each special word count
    for tag in taxonomy:
        df[f'{tag}_count'] = tag_counts.apply(lambda x: x[tag])

    # Calculate total special words per row
    df['total_special_words'] = df[[f'{tag}_count' for tag in taxonomy]].sum(axis=1)
    df['layer_sort'] = df['layer'].str.extract('(\d+)').astype(int)
    df = df.sort_values('layer_sort')

    # Create a long-format dataframe for Altair visualization
    df_long = df.melt(
        id_vars=['layer_sort'],
        value_vars=[f'{word}_count' for word in taxonomy],
        var_name='word_type',
        value_name='count'
    )

    # Clean up the word_type column to remove '_count' suffix
    df_long['word_type'] = df_long['word_type'].str.replace('_count', '')

    y_encoding_line = alt.Y(
            'avg_tags:Q',
            scale=alt.Scale(domain=[1, 3.5]),
            axis=alt.Axis(values=[1,2,3], title=['Size']))
    if hide_y_axis:
        y_encoding_line.axis=alt.Axis(values=[1,2,3], labels=False, ticks=False, title=None, domainColor='lightgray')

    avg_df = (
        df.groupby('layer_sort')['total_special_words']
        .mean()
        .reset_index(name='avg_tags')
    )

    # Create the line chart
    line_chart = alt.Chart(avg_df).mark_line(point=True).encode(
        x=alt.X('layer_sort:Q', axis=None, scale=alt.Scale(domain=[0, df_long['layer_sort'].max()])),
        y=y_encoding_line,
        color=alt.value("grey"),
    ).properties(
        width=width,
        height=35,
    )

    return line_chart

### Make Abstraction Plots

In [11]:
def make_abstraction_plot(model_dfs, p_chart=True, c_chart=True, l_chart=True):
    model_charts = []
    for i, (model_name, df) in enumerate(model_dfs.items()):
        stack = []
        if l_chart:
            stack.append(plot_line_chart(df, primitive_taxonomy, hide_y_axis=i>0))
        if c_chart:
            stack.append(plot_bar_chart(df, combo_taxonomy, combo_colors, normalize=True, show_x=False, show_y=i==0, legend_title='Composition', legend=i==len(model_dfs)-1))
        if p_chart:
            stack.append(plot_bar_chart(df, primitive_taxonomy, primitive_colors, normalize=True, show_y=i==0, legend_title='Component', legend=i==len(model_dfs)-1))

        chart = alt.vconcat(*stack).resolve_scale(x='shared', color='independent').resolve_legend(
            color="independent"
        ).properties(
            title=alt.TitleParams(
                model_name,
                align='center',
                anchor='middle',
                fontSize=12,
                dy=-5
            ),
            bounds='flush',
        )
        model_charts.append(chart)
    return alt.hconcat(*model_charts)

In [12]:
combo_taxonomy = ["none", "quantification", "context", "alternation", "sequence"]
primitive_taxonomy = ["symbol", "lexeme", "field", "context"]

abstraction_plot = make_abstraction_plot(model_dfs).properties(
    title="Semantic Regex Feature Complexity Across Layers",
).configure_title(
    anchor='middle',
    fontSize=15,
    dy=-5,
).configure_axis(
    titleFontSize=11,
    titleColor="grey",
)
abstraction_plot.save(
    'abstraction_plot.png',
    format='png',
    ppi=300,
    scale_factor=2
)
abstraction_plot

/var/folders/gz/zj29llvj3jj52k05h4njylq80000gn/T/ipykernel_98997/2498601034.py:68: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use selection_point instead.
  alt.selection_multi(fields=['word_type'])
/var/folders/gz/zj29llvj3jj52k05h4njylq80000gn/T/ipykernel_98997/2498601034.py:67: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use add_params instead.
  bar_chart = alt.Chart(df_long).mark_bar(width=10).add_selection(
/var/folders/gz/zj29llvj3jj52k05h4njylq80000gn/T/ipykernel_98997/2498601034.py:68: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use selection_point instead.
  alt.selection_multi(fields=['word_type'])
/var/folders/gz/zj29llvj3jj52k05h4njylq80000gn/T/ipykernel_98997/2498601034.py:67: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use add_params instead.
  bar_chart = alt.Chart(df_long).mark_bar(width=10).add_selection(
/var/folders/gz/zj29llvj3jj52k05h4njylq80000gn/T/ipykernel_98997/2498601034.py:68: AltairDeprecation

alt.HConcatChart(...)